SIGNAL TEST

The goal of this section is to add a signal component to the model and get an estimate for the posterior distribution p(mu_s|data). After that we shall proceed with model comparison. 

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

THE CODE BELOW IS A RELOAD OF ALL THE IMPORTANT VARIABLES

In [22]:
bins = pd.read_csv('SourceData/s2_binning_info.csv')
resp_nr = pd.read_csv('SourceData/s2_response_nr.csv')
resp_er = pd.read_csv('SourceData/s2_response_er.csv')
bg = pd.read_csv('SourceData/er_and_cevns_background.csv')
events = pd.read_csv('SourceData/events_after_cuts.csv')

In [23]:
s2_bin_centers_log = bins['log_center_pe'].values
s2_bin_centers_lin = bins['linear_center_pe'].values
s2_bin_widths = (bins['end_pe'] - bins['start_pe']).values
s2_bin_edges = np.concatenate([bins['start_pe'].values, [bins['end_pe'].iloc[-1]]])

In [24]:
#Exctracting information from nuclear recoil response
s2_energies = resp_nr['energy_kev'].values
bin_starts = resp_nr['energy_bin_start_kev'].values
bin_ends = resp_nr['energy_bin_end_kev'].values
dE = bin_ends - bin_starts

response_matrix_nr = resp_nr.values[:,3:] #we start from the 4th column, since the 3 previous ones are energies. The 4th column is s2_bin_000.

In [25]:
import wimprates as wr
reference_cross_section = 1e-45  # cm^2
rate_pertonneyearkev = wr.rate_wimp_std(
    es=s2_energies, 
    mw=4, 
    sigma_nucleon=reference_cross_section)

In [26]:
#Most of the code below was copied directly from the Public Data Release
rate_pertonneyear = rate_pertonneyearkev * dE
rate_before_cutoff = rate_pertonneyear * 0.97678 
#The authors of the paper remove events below 0.7keV
recoil_energy_cutoff_kev = 0.7
rate_after_cutoff = rate_before_cutoff.copy()

# Which bin contains the cutoff?
cutoff_bin_index = (bin_starts < recoil_energy_cutoff_kev).sum() - 1 #This counts how many bins start below 0.7 keV, then subtracts 1 to get the index

# All bins fully below 0.7 keV are removed
rate_after_cutoff[:cutoff_bin_index] = 0

# Suppress the spectrum proportionally in the bin with the cutoff
suppress_by = (
    (recoil_energy_cutoff_kev - bin_starts[cutoff_bin_index]) 
    / bin_starts[cutoff_bin_index])
assert 0 <= suppress_by <= 1

#Only keep the part above the cutoff
rate_after_cutoff[cutoff_bin_index] *= 1 - suppress_by 

In [27]:
b_er = bg['er_background_events']
b_cevns = bg['cevns_background_events']
b_nominal = b_er + b_cevns
k_obs, _ = np.histogram(events['s2_area_pe'].values, bins=s2_bin_edges)

s_i = rate_after_cutoff @ response_matrix_nr
b_i = b_nominal.values

Now we add a signal to the model

In [35]:
#Expected counts per bin including both background and signal
def mu_i(beta, mu_s):
    return beta*b_i + mu_s *s_i




NORA'S CODE BELOW

In [36]:
#nora
from scipy.optimize import minimize_scalar

# signal strength (>0)
mu_s = 1.0   # CHANGE THIS to see effect

#expected function
def mu_i(beta):
    return beta * b_nominal.values + mu_s * s_i

# Likelihood
def log_likelihood_full(beta):
    expected = mu_i(beta)
    expected = np.clip(expected, 1e-12, None)
    return -np.sum(stats.poisson.logpmf(k_obs, expected))


# Fit β
result = minimize_scalar(log_likelihood_full, bounds=(0.1, 30.0), method='bounded')
beta_hat = result.x

# Expected values at best fit
beta_hat_expected = mu_i(beta_hat)

# Plot
fig, ax = plt.subplots(figsize=(9, 4))

ax.step(s2_bin_edges[:-1], k_obs, where='post', color='black', label='Observed data')

ax.step(s2_bin_edges[:-1], beta_hat_expected, where='post',
        color='blue', linestyle='--',
        label=f'Fit (β={beta_hat:.3f}, μ_s={mu_s})')

ax.set_xscale('log')
ax.set_xlabel('S2 area [PE]')
ax.set_ylabel('Events per bin')
ax.set_xlim(90, 3000)
ax.set_xticks([90, 120, 150, 200, 500, 1000, 3000])
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())

ax.legend()
plt.title("Likelihood Function (Signal + Background)")
plt.tight_layout()
plt.show()

NameError: name 'stats' is not defined

In [ ]:
#nora
# region of interest, energy
roi = (165.3, 271.7)
roi_mask = (s2_bin_centers_log >= roi[0]) & (s2_bin_centers_log <= roi[1])


# define μ_i
def mu_i_roi(beta):
    return beta * b_nominal.values[roi_mask] + mu_s * s_i[roi_mask]

# Likelihood
def log_likelihood_full_roi(beta):
    expected = mu_i_roi(beta)
    expected = np.clip(expected, 1e-12, None)
    return -np.sum(stats.poisson.logpmf(k_obs[roi_mask], expected))

# Fit β
result = minimize_scalar(log_likelihood_full_roi, bounds=(0.1, 30.0), method='bounded')
beta_hat = result.x

# Scan β
beta_values = np.linspace(0.1, 30.0, 500)
log_likes = [-log_likelihood_full_roi(b) for b in beta_values]

# Plot
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(beta_values, log_likes, color='blue')
ax.axvline(beta_hat, color='red', linestyle='--', label=f'β̂ = {beta_hat:.3f}')

ax.set_xlabel('β (background normalization)')
ax.set_ylabel('log L(β)')
ax.set_title(f'Log-likelihood vs β (μ_s = {mu_s})')

ax.legend()
plt.tight_layout()
plt.show()

NameError: name 'stats' is not defined